In [4]:
import pandas as pd
from apyori import apriori

dataset = pd.read_csv("market_basket.csv")

display(dataset.head())

,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
0,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,low fat yogurt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
print('Размер датасета:', dataset.shape)
print('Количество пропусков:', dataset.isna().sum().sum())

print('\nТоп-10 товаров:')
print(dataset.stack().value_counts().head(10))

Размер датасета: (7500, 20)
Количество пропусков: 120657

Топ-10 товаров:
mineral water        1787
eggs                 1348
spaghetti            1306
french fries         1282
chocolate            1230
green tea             990
milk                  972
ground beef           737
frozen vegetables     715
pancakes              713
Name: count, dtype: int64


In [6]:
transactions = []

for _, row in dataset.iterrows():
    transaction = [
        str(item).strip()
        for item in row.dropna().tolist()
        if str(item).strip() != ''
    ]

    transactions.append(transaction)

print(transactions[0])
print('Количество транзакций:', len(transactions))

['burgers', 'meatballs', 'eggs']
Количество транзакций: 7500


In [7]:
result = apriori(
    transactions,
    min_support=0.003,
    min_confidence=0.2,
    min_lift=3,
    min_length=2,
    max_length=3
)

association_results = list(result)

print('Количество найденных правил:', len(association_results))

Количество найденных правил: 56


In [8]:
print(association_results[0])

RelationRecord(items=frozenset({'light cream', 'chicken'}), support=0.004533333333333334, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.2905982905982906, lift=4.843304843304844)])


In [9]:
rules = []

for item in association_results:
    for ordered_stat in item.ordered_statistics:
        if len(ordered_stat.items_base) == 0 or len(ordered_stat.items_add) == 0:
            continue

        base_item = ', '.join(sorted([str(x) for x in ordered_stat.items_base]))
        add_item = ', '.join(sorted([str(x) for x in ordered_stat.items_add]))

        rules.append((
            base_item,
            add_item,
            item.support,
            ordered_stat.confidence,
            ordered_stat.lift
        ))

rules_df = pd.DataFrame(
    rules,
    columns=['Base Item', 'Add Item', 'Support', 'Confidence', 'Lift']
)

rules_df[['Support', 'Confidence', 'Lift']] = (
    rules_df[['Support', 'Confidence', 'Lift']].round(4)
)

rules_df.sort_values(by='Lift', ascending=False).head(20)

,Base Item,Add Item,Support,Confidence,Lift
69,"mineral water, whole wheat pasta",olive oil,0.0039,0.4028,6.1275
56,tomato sauce,"ground beef, spaghetti",0.0031,0.2170,5.5352
3,fromage blanc,honey,0.0033,0.2451,5.1781
58,"spaghetti, tomato sauce",ground beef,0.0031,0.4894,4.9799
0,light cream,chicken,0.0045,0.2906,4.8433
2,pasta,escalope,0.0059,0.3729,4.7002
32,"french fries, herb & pepper",ground beef,0.0032,0.4615,4.6968
16,"cereals, spaghetti",ground beef,0.0031,0.4600,4.6811
31,"french fries, ground beef",herb & pepper,0.0032,0.2308,4.6651
8,pasta,shrimp,0.0051,0.3220,4.5145


In [10]:
rules_df.sort_values(by=['Confidence', 'Lift'], ascending=False).head(20)

,Base Item,Add Item,Support,Confidence,Lift
15,"cereals, ground beef",spaghetti,0.0031,0.6765,3.8848
71,"olive oil, tomatoes",spaghetti,0.0044,0.6111,3.5094
57,"ground beef, tomato sauce",spaghetti,0.0031,0.5750,3.3021
26,"cooking oil, ground beef",spaghetti,0.0048,0.5714,3.2816
30,"eggs, red wine",spaghetti,0.0037,0.5283,3.0339
55,"ground beef, shrimp",spaghetti,0.0060,0.5233,3.0049
19,"chicken, olive oil",milk,0.0036,0.5000,3.8580
37,"frozen vegetables, soup",milk,0.0040,0.5000,3.8580
58,"spaghetti, tomato sauce",ground beef,0.0031,0.4894,4.9799
32,"french fries, herb & pepper",ground beef,0.0032,0.4615,4.6968


In [11]:
target_item = 'mineral water'

target_rules = rules_df[
    rules_df['Base Item'].str.contains(target_item, regex=False) |
    rules_df['Add Item'].str.contains(target_item, regex=False)
]

target_rules.sort_values(by='Lift', ascending=False).head(20)

,Base Item,Add Item,Support,Confidence,Lift
69,"mineral water, whole wheat pasta",olive oil,0.0039,0.4028,6.1275
50,"herb & pepper, mineral water",ground beef,0.0067,0.3906,3.9752
59,light cream,"mineral water, spaghetti",0.0032,0.2051,3.4341
68,"mineral water, soup",olive oil,0.0052,0.2254,3.4295
39,"mineral water, shrimp",frozen vegetables,0.0072,0.3068,3.2184


In [12]:
max_lift_target_rule = target_rules.sort_values(
    by='Lift',
    ascending=False
).head(1)

max_lift_target_rule

,Base Item,Add Item,Support,Confidence,Lift
69,"mineral water, whole wheat pasta",olive oil,0.0039,0.4028,6.1275


In [13]:
spaghetti_rules = rules_df[
    rules_df['Base Item'].str.contains('spaghetti', regex=False) |
    rules_df['Add Item'].str.contains('spaghetti', regex=False)
]

spaghetti_rules.sort_values(by='Lift', ascending=False).head(20)

,Base Item,Add Item,Support,Confidence,Lift
56,tomato sauce,"ground beef, spaghetti",0.0031,0.2170,5.5352
58,"spaghetti, tomato sauce",ground beef,0.0031,0.4894,4.9799
16,"cereals, spaghetti",ground beef,0.0031,0.4600,4.6811
51,"herb & pepper, spaghetti",ground beef,0.0064,0.3934,4.0038
15,"cereals, ground beef",spaghetti,0.0031,0.6765,3.8848
71,"olive oil, tomatoes",spaghetti,0.0044,0.6111,3.5094
67,"spaghetti, whole wheat pasta",milk,0.0040,0.4545,3.5073
45,"frozen vegetables, spaghetti",tomatoes,0.0067,0.2392,3.4976
54,"pepper, spaghetti",ground beef,0.0033,0.3378,3.4380
59,light cream,"mineral water, spaghetti",0.0032,0.2051,3.4341


In [14]:
chocolate_rules = rules_df[
    rules_df['Base Item'].str.contains('chocolate', regex=False) |
    rules_df['Add Item'].str.contains('chocolate', regex=False)
]

chocolate_rules.sort_values(by='Lift', ascending=False).head(20)

,Base Item,Add Item,Support,Confidence,Lift
24,"chocolate, herb & pepper",ground beef,0.0040,0.4412,4.4896
22,"chocolate, frozen vegetables",shrimp,0.0053,0.2326,3.2602
23,"chocolate, shrimp",frozen vegetables,0.0053,0.2963,3.1080
11,"chocolate, turkey",burgers,0.0031,0.2706,3.1031
25,"chocolate, soup",milk,0.0040,0.3947,3.0458


In [15]:
print('Всего правил:', len(rules_df))
print('Правил, связанных с mineral water:', len(target_rules))
print('Правил, связанных со spaghetti:', len(spaghetti_rules))
print('Правил, связанных с chocolate:', len(chocolate_rules))

row = max_lift_target_rule.iloc[0]
print()
print('Правило с максимальным lift среди правил, связанных с mineral water:')
print(f"{row['Base Item']} -> {row['Add Item']}")
print(f"Support: {row['Support']}")
print(f"Confidence: {row['Confidence']}")
print(f"Lift: {row['Lift']}")

Всего правил: 74
Правил, связанных с mineral water: 5
Правил, связанных со spaghetti: 26
Правил, связанных с chocolate: 5

Правило с максимальным lift среди правил, связанных с mineral water:
mineral water, whole wheat pasta -> olive oil
Support: 0.0039
Confidence: 0.4028
Lift: 6.1275
